<a href="https://colab.research.google.com/github/swadhin94/Python-Test/blob/main/Amazon%20job%20scrapping%202.0%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install selenium pandas beautifulsoup4 openpyxl webdriver-manager

# Install Chrome browser using direct .deb package download
!apt-get update
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -fy
!apt --fix-broken install
!apt-get autoremove

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# Setup Chrome options for headless mode
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.binary_location = '/usr/bin/google-chrome'

# Initialize ChromeDriver using webdriver_manager
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.get("https://www.amazon.jobs/en/search?offset=0&result_limit=10&sort=relevant&category%5B%5D=project-program-product-management-non-tech&country%5B%5D=IND&distanceType=Mi&radius=24km&industry_experience=seven_plus_years&latitude=&longitude=&loc_group_id=&loc_query=&base_query=product&city=&country=&region=&county=&query_options=&")

time.sleep(5)

jobs = driver.find_elements(By.CSS_SELECTOR, "a.job-link")

job_links = [job.get_attribute("href") for job in jobs]

print(job_links[:10])
driver.quit()

# The following lines related to scrolling and pagination will also cause a SessionNotCreatedException
# if the driver is quit before they are called. I'm commenting them out for now to focus on the main error.
# driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
# time.sleep(3)

# while True:
#     try:
#         next_button = driver.find_element(By.CSS_SELECTOR, ".pagination-next")
#         next_button.click()
#         time.sleep(3)
#     except:
#         break

def classify_job(desc):
    desc = desc.lower()

    agile = "y" if any(word in desc for word in ["agile", "scrum", "sprint"]) else "n"
    pm = "y" if any(word in desc for word in ["program management", "project management"]) else "y"
    mba = "y" if "mba" in desc else "n"

    return agile, pm, mba

import pandas as pd

data = []

# The function scrape_job_details is not defined. This will cause a NameError.
# It needs to be defined to proceed with scraping details for each job link.
# For now, I'm just creating an empty dataframe or skipping this part until scrape_job_details is implemented.
# For demonstration, I'll create an empty dataframe if job_links is empty or if scrape_job_details is not available.
if not job_links:
    print("No job links found or unable to scrape details without 'scrape_job_details' function.")
    df = pd.DataFrame(columns=[
        "Title","Job ID","Description","Agile","PM","MBA"
    ])
else:
    # Assuming scrape_job_details would be defined later, for now we will skip processing links if it's missing.
    print("To process job details, the 'scrape_job_details' function needs to be defined.")
    df = pd.DataFrame(columns=[
        "Title","Job ID","Description","Agile","PM","MBA"
    ])

# The original loop would cause a NameError if scrape_job_details is not defined:
# for link in job_links:
#     try:
#         title, job_id, desc = scrape_job_details(link)
#         agile, pm, mba = classify_job(desc)
#
#         data.append([title, job_id, desc[:300], agile, pm, mba])
#     except:
#         continue

# df = pd.DataFrame(data, columns=[
#     "Title","Job ID","Description","Agile","PM","MBA"
# ])

df.to_csv("amazon_jobs.csv", index=False)

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-03-29 05:48:26--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 142.251.163.93, 142.251.163.91, 142.251.163.190, 

In [9]:
"""
Amazon Jobs Scraper – Full Pagination Edition
Scrapes ALL PM/PPM jobs in India and saves to CSV.

Install once:
    pip install selenium pandas beautifulsoup4 webdriver-manager
    apt-get update && wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -fy
"""

import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# ── Constants ─────────────────────────────────────────────────────────────────

RESULTS_PER_PAGE = 10          # Amazon Jobs returns 10 per page by default
PAGE_LOAD_WAIT   = 12          # seconds to wait for job cards to appear
CRAWL_DELAY      = 1.5         # polite delay between requests (seconds)
OUTPUT_CSV       = "amazon_jobs.csv"

SEARCH_URL = (
    "https://www.amazon.jobs/en/search"
    "?offset={offset}"
    "&result_limit={limit}"
    "&sort=relevant"
    "&category[]=project-program-product-management-non-tech"
    "&country[]=IND"
    "&distanceType=Mi&radius=24km"
    "&industry_experience=seven_plus_years"
    "&base_query=product"
)


# ── Driver ────────────────────────────────────────────────────────────────────

def create_driver() -> webdriver.Chrome:
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.binary_location = "/usr/bin/google-chrome"
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )


# ── Step 1: Collect ALL job links via offset-based pagination ─────────────────

def collect_all_job_links(driver: webdriver.Chrome) -> list[str]:
    """
    Paginate through every offset (0, 10, 20, ...) until a page returns 0 links.
    Driving by offset is more reliable than clicking the Next button.
    """
    all_links: list[str] = []
    seen:      set[str]  = set()
    offset = 0
    page   = 1

    print("=" * 60)
    print("PHASE 1 — Collecting job links")
    print("=" * 60)

    while True:
        url = SEARCH_URL.format(offset=offset, limit=RESULTS_PER_PAGE)
        print(f"\n  Page {page:>3}  (offset={offset})")
        driver.get(url)

        # Wait for job cards; timeout means no more results
        try:
            WebDriverWait(driver, PAGE_LOAD_WAIT).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a.job-link"))
            )
        except Exception:
            print(f"  -> No job cards found after {PAGE_LOAD_WAIT}s — pagination complete.")
            break

        page_links = [
            el.get_attribute("href")
            for el in driver.find_elements(By.CSS_SELECTOR, "a.job-link")
            if el.get_attribute("href")
        ]

        new_links = [l for l in page_links if l not in seen]
        if not new_links:
            print("  -> No new links on this page — pagination complete.")
            break

        seen.update(new_links)
        all_links.extend(new_links)
        print(f"  -> {len(new_links)} new links  |  running total: {len(all_links)}")

        # Partial page == last page
        if len(page_links) < RESULTS_PER_PAGE:
            print("  -> Partial page — this is the last page.")
            break

        offset += RESULTS_PER_PAGE
        page   += 1
        time.sleep(CRAWL_DELAY)

    print(f"\n  Total unique job links collected: {len(all_links)}")
    return all_links


# ── Step 2: Scrape each job detail page ───────────────────────────────────────

def scrape_job_details(driver: webdriver.Chrome, url: str) -> dict:
    """
    Visit a single job page and return title, job_id, location,
    posted_date, and full description.
    """
    empty = {
        "title": "", "job_id": "", "location": "",
        "posted_date": "", "description": ""
    }

    driver.get(url)

    try:
        WebDriverWait(driver, PAGE_LOAD_WAIT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1.title"))
        )
    except Exception:
        return empty

    def safe(selector: str) -> str:
        try:
            return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
        except Exception:
            return ""

    title       = safe("h1.title")
    location    = safe(".location-and-id .location")
    posted_date = safe(".posted-date")

    # Job ID: page label first, then fall back to URL digits
    job_id = safe(".job-id")
    if not job_id:
        parts  = [p for p in url.rstrip("/").split("/") if p.isdigit()]
        job_id = parts[-1] if parts else ""

    description = safe(".job-description")

    return {
        "title":       title,
        "job_id":      job_id,
        "location":    location,
        "posted_date": posted_date,
        "description": description,
    }


# ── Step 3: Classify each job description ────────────────────────────────────

def classify_job(description: str) -> dict:
    """
    Tag a job description. Each field returns 'Y' or 'N'.

    Bug fixed from v1: pm else-branch was hardcoded 'y' so every
    job was marked as requiring PM experience. Now defaults to 'N'.
    """
    desc = description.lower()
    return {
        "agile": "Y" if any(kw in desc for kw in
                            ["agile", "scrum", "sprint", "kanban"]) else "N",
        "pm":    "Y" if any(kw in desc for kw in
                            ["program management", "project management",
                             "product management"]) else "N",
        "mba":   "Y" if "mba" in desc else "N",
    }


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    driver  = create_driver()
    records = []

    try:
        # Phase 1 — collect every job URL
        job_links = collect_all_job_links(driver)

        # Phase 2 — visit each job page and extract data
        print("\n" + "=" * 60)
        print("PHASE 2 — Scraping job detail pages")
        print("=" * 60)

        for i, link in enumerate(job_links, start=1):
            print(f"  [{i:>3}/{len(job_links)}]  {link}")
            try:
                details = scrape_job_details(driver, link)
                tags    = classify_job(details["description"])
                records.append({
                    "Title":        details["title"],
                    "Job ID":       details["job_id"],
                    "Location":     details["location"],
                    "Posted Date":  details["posted_date"],
                    "URL":          link,
                    "Description":  details["description"][:600],
                    "Agile":        tags["agile"],
                    "PM":           tags["pm"],
                    "MBA":          tags["mba"],
                })
            except Exception as exc:
                print(f"    X  Error scraping {link}: {exc}")

            time.sleep(CRAWL_DELAY)

    finally:
        driver.quit()

    # Save to CSV (utf-8-sig so Excel opens it correctly without BOM issues)
    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 60)
    print(f"  Done!  {len(df)} jobs saved to '{OUTPUT_CSV}'")
    print("=" * 60)
    if not df.empty:
        print(df[["Title", "Location", "Agile", "PM", "MBA"]].to_string(index=False))
    return df


if __name__ == "__main__":
    df = main()

PHASE 1 — Collecting job links

  Page   1  (offset=0)
  -> 10 new links  |  running total: 10

  Page   2  (offset=10)
  -> 10 new links  |  running total: 20

  Page   3  (offset=20)
  -> 10 new links  |  running total: 30

  Page   4  (offset=30)
  -> 10 new links  |  running total: 40

  Page   5  (offset=40)
  -> 10 new links  |  running total: 50

  Page   6  (offset=50)
  -> 10 new links  |  running total: 60

  Page   7  (offset=60)
  -> 10 new links  |  running total: 70

  Page   8  (offset=70)
  -> 10 new links  |  running total: 80

  Page   9  (offset=80)
  -> 10 new links  |  running total: 90

  Page  10  (offset=90)
  -> 10 new links  |  running total: 100

  Page  11  (offset=100)
  -> 5 new links  |  running total: 105
  -> Partial page — this is the last page.

  Total unique job links collected: 105

PHASE 2 — Scraping job detail pages
  [  1/105]  https://www.amazon.jobs/en/jobs/3142778/senior-product-manager
  [  2/105]  https://www.amazon.jobs/en/jobs/3142769/sen